In [1]:
from pathlib import Path

DIR_VIDEOS = Path(r"C:\Users\isabe\signapp\P_09")
DIR_FRAMES = Path(r"C:\Users\isabe\signapp\frames")
DIR_KEYPOINTS = Path(r"C:\Users\isabe\signapp\keypoints")
DIR_DATASET = Path(r"C:\Users\isabe\signapp\split_dataset")

DIR_FRAMES.mkdir(parents=True, exist_ok=True)
DIR_KEYPOINTS.mkdir(parents=True, exist_ok=True)
DIR_DATASET.mkdir(parents=True, exist_ok=True)

VIDEO_EXTENSIONS = {".mp4", ".avi", ".mov", ".mkv"}


In [2]:
import cv2

video_paths = sorted(
    p for p in DIR_VIDEOS.rglob("*") if p.suffix.lower() in VIDEO_EXTENSIONS
)
print(f"Found {len(video_paths)} videos in {DIR_VIDEOS}")

for video_path in video_paths:
    out_dir = DIR_FRAMES / video_path.relative_to(DIR_VIDEOS).with_suffix("")
    out_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    frame_idx = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.resize(frame, None, fx=0.75, fy=0.75, interpolation=cv2.INTER_AREA)
        frame_path = out_dir / f"{video_path.stem}_{frame_idx:05d}.jpg"
        cv2.imwrite(str(frame_path), frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
        frame_idx += 1
    cap.release()

    print(f"{video_path.relative_to(DIR_VIDEOS)}: saved {frame_idx} frames -> {out_dir}")


Found 100 videos in C:\Users\isabe\signapp\P_09
P_09\a_veces\a_veces09_1.mp4: saved 88 frames -> C:\Users\isabe\signapp\frames\P_09\a_veces\a_veces09_1
P_09\a_veces\a_veces09_10.mp4: saved 112 frames -> C:\Users\isabe\signapp\frames\P_09\a_veces\a_veces09_10
P_09\a_veces\a_veces09_2.mp4: saved 121 frames -> C:\Users\isabe\signapp\frames\P_09\a_veces\a_veces09_2
P_09\a_veces\a_veces09_3.mp4: saved 118 frames -> C:\Users\isabe\signapp\frames\P_09\a_veces\a_veces09_3
P_09\a_veces\a_veces09_4.mp4: saved 91 frames -> C:\Users\isabe\signapp\frames\P_09\a_veces\a_veces09_4
P_09\a_veces\a_veces09_5.mp4: saved 104 frames -> C:\Users\isabe\signapp\frames\P_09\a_veces\a_veces09_5
P_09\a_veces\a_veces09_6.mp4: saved 106 frames -> C:\Users\isabe\signapp\frames\P_09\a_veces\a_veces09_6
P_09\a_veces\a_veces09_7.mp4: saved 121 frames -> C:\Users\isabe\signapp\frames\P_09\a_veces\a_veces09_7
P_09\a_veces\a_veces09_8.mp4: saved 103 frames -> C:\Users\isabe\signapp\frames\P_09\a_veces\a_veces09_8
P_09\a_

In [ ]:
import numpy as np
import mediapipe as mp

mp_holistic = mp.solutions.holistic

# Reduced face-mesh points (indices into MediaPipe's 468-point face mesh):
# 2 mouth corners, 2 eye corners, 2 eyebrows, 2 nose points, top of face,
# forehead, and chin.
FACE_LANDMARK_IDS = [
    61, 291,   # mouth: left / right corner
    33, 263,   # eyes: right / left outer corner
    105, 334,  # eyebrows: right / left
    1, 6,      # nose: tip / bridge
    10,        # top of face
    9,         # forehead
    152,       # chin (bottom of face)
]


def extract_keypoints(results):
    if results.pose_landmarks:
        pose = np.array(
            [[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark]
        ).flatten()
    else:
        pose = np.zeros(33 * 4)

    if results.face_landmarks:
        face_lms = results.face_landmarks.landmark
        face = np.array([[face_lms[i].x, face_lms[i].y, face_lms[i].z] for i in FACE_LANDMARK_IDS]).flatten()
    else:
        face = np.zeros(len(FACE_LANDMARK_IDS) * 3)

    if results.left_hand_landmarks:
        lh = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark]).flatten()
    else:
        lh = np.zeros(21 * 3)

    if results.right_hand_landmarks:
        rh = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark]).flatten()
    else:
        rh = np.zeros(21 * 3)

    return np.concatenate([pose, face, lh, rh])


frame_dirs = sorted({p.parent for p in DIR_FRAMES.rglob("*.jpg")})

with mp_holistic.Holistic(static_image_mode=True) as holistic:
    for frame_dir in frame_dirs:
        out_dir = DIR_KEYPOINTS / frame_dir.relative_to(DIR_FRAMES)
        out_dir.mkdir(parents=True, exist_ok=True)

        frame_paths = sorted(frame_dir.glob("*.jpg"))
        for frame_path in frame_paths:
            image = cv2.imread(str(frame_path))
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            results = holistic.process(image_rgb)

            keypoints = extract_keypoints(results)
            np.save(out_dir / f"{frame_path.stem}.npy", keypoints)

        print(f"{frame_dir.relative_to(DIR_FRAMES)}: saved {len(frame_paths)} keypoint files -> {out_dir}")


AttributeError: module 'mediapipe' has no attribute 'solutions'